Setup

In [27]:
# Install dependencies (Colab has networkx, matplotlib)
!pip install networkx matplotlib numpy -q

# Import required libraries

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

from dataclasses import dataclass
from copy import deepcopy
from typing import Dict, List, Tuple, Optional

print("ZX Pipeline Ready!")

ZX Pipeline Ready!


In [28]:
# ZX constants

Z_SPIDER = "Z"
X_SPIDER = "X"

NORMAL_EDGE = "normal"
HADAMARD_EDGE = "hadamard"


print("ZX Calculus Optimizer Setup Complete")

ZX Calculus Optimizer Setup Complete


ZX Graph Data Structures

In [29]:
@dataclass
class Spider:
    id: int
    color: str
    phase: float = 0.0

    def __repr__(self):
        return f"Spider(id={self.id}, color={self.color}, phase={self.phase})"


class ZXGraph:
    def __init__(self):
        self.graph = nx.Graph()
        self.spiders = {}
        self.next_id = 0


    def add_spider(self, color, phase=0.0):
        spider_id = self.next_id
        self.next_id += 1

        spider = Spider(
            id=spider_id,
            color=color,
            phase=phase
        )

        self.spiders[spider_id] = spider

        self.graph.add_node(
            spider_id,
            color=color,
            phase=phase
        )

        return spider_id


    def add_edge(self, spider1, spider2, edge_type="normal"):
        self.graph.add_edge(
            spider1,
            spider2,
            type=edge_type
        )


    def remove_spider(self, spider_id):
        if spider_id in self.spiders:
            del self.spiders[spider_id]

        if spider_id in self.graph:
            self.graph.remove_node(spider_id)


    def get_spider(self, spider_id):
        return self.spiders.get(spider_id)


    def neighbors(self, spider_id):
        return list(self.graph.neighbors(spider_id))


    def number_of_spiders(self):
        return self.graph.number_of_nodes()


    def number_of_edges(self):
        return self.graph.number_of_edges()


    def copy(self):
        return deepcopy(self)


    def summary(self):
        print("Spiders:", self.number_of_spiders())
        print("Edges:", self.number_of_edges())

        for node in self.graph.nodes:
            data = self.graph.nodes[node]
            print(
                node,
                "Color:",
                data["color"],
                "Phase:",
                data["phase"]
            )


print("ZXGraph class created successfully")

ZXGraph class created successfully


In [30]:
# Test ZXGraph creation

zx = ZXGraph()

s1 = zx.add_spider("Z", phase=0)
s2 = zx.add_spider("Z", phase=np.pi/4)
s3 = zx.add_spider("X", phase=np.pi)

zx.add_edge(s1, s2)
zx.add_edge(s2, s3)


zx.summary()

Spiders: 3
Edges: 2
0 Color: Z Phase: 0
1 Color: Z Phase: 0.7853981633974483
2 Color: X Phase: 3.141592653589793


Example Quantum Circuits

In [31]:
class QuantumCircuit:
    def __init__(self, qubits):
        self.qubits = qubits
        self.operations = []


    def add_gate(self, gate, qubit, target=None, phase=None):
        operation = {
            "gate": gate,
            "qubit": qubit,
            "target": target,
            "phase": phase
        }

        self.operations.append(operation)


    def display(self):
        for op in self.operations:
            print(op)



# Circuit 1: H - CNOT - T

circuit1 = QuantumCircuit(2)

circuit1.add_gate("H", 0)
circuit1.add_gate("CNOT", 0, target=1)
circuit1.add_gate("T", 0, phase=np.pi/4)



# Circuit 2: Clifford circuit

circuit2 = QuantumCircuit(1)

circuit2.add_gate("H", 0)
circuit2.add_gate("S", 0, phase=np.pi/2)
circuit2.add_gate("H", 0)



# Circuit 3: Non-Clifford circuit

circuit3 = QuantumCircuit(1)

circuit3.add_gate("T", 0, phase=np.pi/4)
circuit3.add_gate("H", 0)
circuit3.add_gate("Tdg", 0, phase=-np.pi/4)



print("Example circuits created")

Example circuits created


In [32]:
# Display circuits

print("Circuit 1")
circuit1.display()

print("\nCircuit 2")
circuit2.display()

print("\nCircuit 3")
circuit3.display()

Circuit 1
{'gate': 'H', 'qubit': 0, 'target': None, 'phase': None}
{'gate': 'CNOT', 'qubit': 0, 'target': 1, 'phase': None}
{'gate': 'T', 'qubit': 0, 'target': None, 'phase': 0.7853981633974483}

Circuit 2
{'gate': 'H', 'qubit': 0, 'target': None, 'phase': None}
{'gate': 'S', 'qubit': 0, 'target': None, 'phase': 1.5707963267948966}
{'gate': 'H', 'qubit': 0, 'target': None, 'phase': None}

Circuit 3
{'gate': 'T', 'qubit': 0, 'target': None, 'phase': 0.7853981633974483}
{'gate': 'H', 'qubit': 0, 'target': None, 'phase': None}
{'gate': 'Tdg', 'qubit': 0, 'target': None, 'phase': -0.7853981633974483}


Circuit to ZX Diagram Conversion

In [33]:
def circuit_to_zx(circuit):
    zx = ZXGraph()

    qubit_wires = {}

    for q in range(circuit.qubits):
        wire = zx.add_spider(
            color=Z_SPIDER,
            phase=0
        )
        qubit_wires[q] = wire


    for operation in circuit.operations:

        gate = operation["gate"]
        q = operation["qubit"]
        current = qubit_wires[q]


        if gate == "H":

            h_spider = zx.add_spider(
                color=X_SPIDER,
                phase=0
            )

            zx.add_edge(
                current,
                h_spider,
                HADAMARD_EDGE
            )

            qubit_wires[q] = h_spider



        elif gate == "T":

            t_spider = zx.add_spider(
                color=Z_SPIDER,
                phase=np.pi/4
            )

            zx.add_edge(
                current,
                t_spider
            )

            qubit_wires[q] = t_spider



        elif gate == "Tdg":

            t_spider = zx.add_spider(
                color=Z_SPIDER,
                phase=-np.pi/4
            )

            zx.add_edge(
                current,
                t_spider
            )

            qubit_wires[q] = t_spider



        elif gate == "S":

            s_spider = zx.add_spider(
                color=Z_SPIDER,
                phase=np.pi/2
            )

            zx.add_edge(
                current,
                s_spider
            )

            qubit_wires[q] = s_spider



        elif gate == "CNOT":

            control = q
            target = operation["target"]


            control_spider = zx.add_spider(
                color=Z_SPIDER,
                phase=0
            )

            target_spider = zx.add_spider(
                color=X_SPIDER,
                phase=0
            )


            zx.add_edge(
                qubit_wires[control],
                control_spider
            )

            zx.add_edge(
                qubit_wires[target],
                target_spider
            )


            zx.add_edge(
                control_spider,
                target_spider
            )


            qubit_wires[control] = control_spider
            qubit_wires[target] = target_spider


    return zx



print("Circuit to ZX conversion function created")

Circuit to ZX conversion function created


In [34]:
# Convert example circuits

zx1 = circuit_to_zx(circuit1)
zx2 = circuit_to_zx(circuit2)
zx3 = circuit_to_zx(circuit3)


print("Circuit 1 ZX Graph")
zx1.summary()


print("\nCircuit 2 ZX Graph")
zx2.summary()


print("\nCircuit 3 ZX Graph")
zx3.summary()

Circuit 1 ZX Graph
Spiders: 6
Edges: 5
0 Color: Z Phase: 0
1 Color: Z Phase: 0
2 Color: X Phase: 0
3 Color: Z Phase: 0
4 Color: X Phase: 0
5 Color: Z Phase: 0.7853981633974483

Circuit 2 ZX Graph
Spiders: 4
Edges: 3
0 Color: Z Phase: 0
1 Color: X Phase: 0
2 Color: Z Phase: 1.5707963267948966
3 Color: X Phase: 0

Circuit 3 ZX Graph
Spiders: 4
Edges: 3
0 Color: Z Phase: 0
1 Color: Z Phase: 0.7853981633974483
2 Color: X Phase: 0
3 Color: Z Phase: -0.7853981633974483
